### Pipeline :

It combines multiple steps like preprocessing and algorithm.

It executes the steps one by one.

#### Syntax :

from sklearn.pipeline import Pipeline


main_pipeline=pipeline(steps=[('name',step),,,,,('name',step)])
                

main_pipeline.fit(xtrain ,ytrain)

In [1]:
# Scenario 1

# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.tree import DecisionTreeClassifierer

# pipeline = Pipeline([
#     ('scaler', StandardScaler()),
#     ('classifier', DecisionTreeClassifier(max_depth=5))
# ])

# pipeline.fit(X_train, y_train)

# y_pred = pipeline.predict(X_test)

* Here we are using the pipeline to preprocess the data and train the model in one step,
* which simplifies the workflow and ensures that the same preprocessing steps are applied to both the training and test data.
* When we use the single preprocessing technique the pipeline apply the same preprocessing techniques to all feature columns,
* but in some cases, we may want to apply different preprocessing techniques to different feature columns.
* There we can use the ColumnTransformer to apply different preprocessing techniques to different feature columns.

In [2]:
# Scenario 2

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('name1', transformer1, column_list1),
#         ('name2', transformer2, column_list2),
#         ...
#     ],
#     remainder='drop'   # or 'passthrough'
# )

# main_pipeline = Pipeline(
#     steps=[
#         ('pre',preprocessor),
#         ('model',Algorithm())
#     ]
# )

* step_name → Any name you choose (e.g., 'num', 'cat').
* transformer → Preprocessing method (e.g., StandardScaler(), OneHotEncoder()).
* columns → List of column names or column indices.
* remainder
    * 'drop' → Drops columns not specified (default).
    * 'passthrough' → Keeps remaining columns unchanged.

* Should not ues same column on to multiple preprocessing technique, It will generate the duplicate column which reduces the accuracy

In [3]:
# Scenario 3

# preprocessor = ColumnTransformer(
#     transformers=[
#         (
#             'numer',
#             Pipeline(
#                 steps=[
#                     ('scaling', StandardScaler()),
#                     ('poly', PolynomialFeatures(degree=2, include_bias=False))
#                 ]
#             ),
#             num_cols
#         ),
#         (
#             'encodes',
#             OneHotEncoder(handle_unknown='ignore'),
#             cat_cols
#         )
#     ]
# )

#### This setup is useful when:

* When dataset has both numerical and categorical features.
* When we want to create polynomial features only for the numerical columns.
* When we want the categorical columns to be one-hot encoded.
* When we want all preprocessing to happen automatically during both training and prediction.

### ML Pipeline Implementation 
#### >. ML Algorithm : Decision Tree(Classifier)
#### >. Dataset :  Telco Customer Churn

In [4]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv('telco.csv')

In [6]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [8]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [9]:
(df["TotalCharges"] == " ").sum()

np.int64(11)

1. Customer ID

* Every customer has a unique ID.
* It provides no predictive value, so drop it

2. Total Charges

* there are 11 blank values (" "), so convert them to NaN before changing the type.

df["TotalCharges"] = (
    df["TotalCharges"]
    .replace(" ", np.nan)
    .astype(float)
)

3. Churn(target)

* The target column is text. Decision trees need numeric targets.

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

In [10]:
# 2 Total Charges have empty values. We will replace them with NaN and then drop the rows with NaN values.

df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])

df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [11]:
# 1. Drop the customerID column as it is not useful for our analysis.

df.drop("customerID", axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   object 
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   object 
 3   Dependents        7032 non-null   object 
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   object 
 6   MultipleLines     7032 non-null   object 
 7   InternetService   7032 non-null   object 
 8   OnlineSecurity    7032 non-null   object 
 9   OnlineBackup      7032 non-null   object 
 10  DeviceProtection  7032 non-null   object 
 11  TechSupport       7032 non-null   object 
 12  StreamingTV       7032 non-null   object 
 13  StreamingMovies   7032 non-null   object 
 14  Contract          7032 non-null   object 
 15  PaperlessBilling  7032 non-null   object 
 16  PaymentMethod     7032 non-null   object 
 17  

In [12]:
# 3. Convert the target variable "Churn" into a binary variable (0 for No, 1 for Yes).

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

df.head(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


### Split Features and Target

In [13]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

### Train-Test Split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

### Identify Feature Types

In [15]:
categorical_cols = X_train.select_dtypes(include="object").columns

numeric_cols = X_train.select_dtypes(exclude="object").columns

In [16]:
from sklearn.pipeline import Pipeline

In [17]:
from sklearn.preprocessing import OneHotEncoder

In [18]:
from sklearn.tree import DecisionTreeClassifier

In [19]:
from sklearn.compose import ColumnTransformer

### Pipeline

In [20]:
# WE  NEED TO ENCODE THE CATEGORICAL COLUMNS USING "ONEHOTENCODER" AND THEN CHOOSE MODEL AS DECISION TREE CLASSIFIER

# PIPELINE([
#     ('ONEHOTENCODING',OneHotEncoder(handle_unknown="ignore")) -----> Here One Hot encoder will encode all columns not only Categorical
#     ('MODEL', DecisionTreeClassifier(random_state=42))       |---> so this is Wrong, we need to use Column_Transformer to separately encode the categarical columns
# ])

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

### Pipeline
AFTER ENCODING USING COLUMN TRANSFORMER

In [22]:

main_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(random_state=42))
])

In [23]:

main_pipeline.fit(X_train, y_train)

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\compose\_column_transformer.py:1651: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', DecisionTreeClassifier(random_state=42))])

In [24]:
y_train_pred = main_pipeline.predict(X_train)
y_test_pred = main_pipeline.predict(X_test)

In [25]:
from sklearn.metrics import classification_report

In [26]:
print(classification_report(y_train, y_train_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4130
           1       1.00      0.99      1.00      1495

    accuracy                           1.00      5625
   macro avg       1.00      1.00      1.00      5625
weighted avg       1.00      1.00      1.00      5625



In [27]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.81      0.80      0.80      1033
           1       0.46      0.47      0.47       374

    accuracy                           0.71      1407
   macro avg       0.63      0.64      0.63      1407
weighted avg       0.71      0.71      0.71      1407



In [28]:
print(main_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', DecisionTreeClassifier(random_state=42))])


In [29]:
print(preprocessor)

ColumnTransformer(remainder='passthrough',
                  transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])
